In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import copernicusmarine
from IPython.display import display

BASE = Path("experiment_v2") if Path("experiment_v2/data").is_dir() else Path(".")
ARCHIVE_DIR = BASE / "data/raw/nrt_archive"

REGION = {
    "minimum_longitude": 13.90,
    "maximum_longitude": 15.20,
    "minimum_latitude": 35.55,
    "maximum_latitude": 36.90,
}

SOURCES = {
    "chl": {
        "dataset_id": "cmems_obs-oc_med_bgc-plankton_nrt_l3-multi-1km_P1D",
        "variable": "CHL",
        "margin": 0.0,
        "recent_days": None,      # keep every day the rolling window offers
    },
    "chl_l4": {
        "dataset_id": "cmems_obs-oc_med_bgc-plankton_nrt_l4-gapfree-multi-1km_P1D",
        "variable": "CHL",
        "margin": 0.0,
        "recent_days": None,
    },
    "sst": {
        "dataset_id": "SST_MED_SST_L3S_NRT_OBSERVATIONS_010_012_a",
        "variable": "adjusted_sea_surface_temperature",
        "margin": 0.07,           # same as NB07: covers the edge SST cells
        "recent_days": 30,        # the SST record goes back to 2008
    },
}


In [2]:
remote = {}
rows = []

for name, source in SOURCES.items():
    catalogue = copernicusmarine.describe(dataset_id=source["dataset_id"])
    dataset = next(
        dataset
        for product in catalogue.products
        for dataset in product.datasets
        if dataset.dataset_id == source["dataset_id"]
    )
    version = dataset.versions[0].label

    data = copernicusmarine.open_dataset(
        dataset_id=source["dataset_id"],
        dataset_version=version,
        variables=[source["variable"]],
        minimum_longitude=REGION["minimum_longitude"] - source["margin"],
        maximum_longitude=REGION["maximum_longitude"] + source["margin"],
        minimum_latitude=REGION["minimum_latitude"] - source["margin"],
        maximum_latitude=REGION["maximum_latitude"] + source["margin"],
    )
    data = data.rename({
        old: new
        for old, new in [("lat", "latitude"), ("lon", "longitude")]
        if old in data.coords and new not in data.coords
    })

    remote[name] = {"data": data, "version": version}
    times = pd.DatetimeIndex(data.time.values)
    rows.append({
        "Source": name,
        "Version": version,
        "Days offered": len(times),
        "First": f"{times.min():%Y-%m-%d}",
        "Latest": f"{times.max():%Y-%m-%d}",
    })

display(pd.DataFrame(rows).set_index("Source"))


Fetching catalogue 1: 100%|██████████| 2/2 [00:01<00:00,  1.11it/s]
INFO - 2026-10-19T15:17:23Z - Selected dataset version: "202211"
INFO - 2026-10-19T15:17:23Z - Selected dataset part: "default"
Fetching catalogue 1: 100%|██████████| 2/2 [00:01<00:00,  1.05it/s]
INFO - 2026-10-19T15:17:27Z - Selected dataset version: "202207"
INFO - 2026-10-19T15:17:27Z - Selected dataset part: "default"
WARNING - 2026-10-19T15:17:27Z - You are using the dataset cmems_obs-oc_med_bgc-plankton_nrt_l4-gapfree-multi-1km_P1D, version '202207', part 'default'. This exact version and part of the dataset will be retired on the 2027-01-12T13:00:00.000Z. For more information you can check: https://marine.copernicus.eu/user-corner/product-roadmap/transition-information
Fetching catalogue 1: 100%|██████████| 2/2 [00:00<00:00,  3.20it/s]
INFO - 2026-10-19T15:17:30Z - Selected dataset version: "202311"
INFO - 2026-10-19T15:17:30Z - Selected dataset part: "default"
WARNING - 2026-10-19T15:17:30Z - You are using the 

,Version,Days offered,First,Latest
Source,,,,
chl,202211,9,2026-10-10,2026-10-18
chl_l4,202207,12,2026-10-07,2026-10-18
sst,202311,6867,2008-01-01,2026-10-19


In [3]:
retrieved_utc = pd.Timestamp.now(tz="UTC")
rows = []

for name, source in SOURCES.items():
    data = remote[name]["data"]
    variable = source["variable"]

    days = pd.DatetimeIndex(data.time.values)
    if source["recent_days"]:
        newest = days.max().normalize()
        days = days[days >= newest - pd.Timedelta(days=source["recent_days"] - 1)]

    out_dir = ARCHIVE_DIR / name
    out_dir.mkdir(parents=True, exist_ok=True)

    for day in days:
        path = out_dir / f"{day:%Y-%m-%d}.nc"
        row = {"Source": name, "Date": f"{day:%Y-%m-%d}"}

        # Keep the first copy we retrieved: that is what a forecast would have used.
        if path.exists():
            rows.append({**row, "Status": "Already archived"})
            continue

        field = data[[variable]].sel(time=[day]).load()
        valid = int(np.isfinite(field[variable]).sum().item())

        field.attrs.update({
            "source_dataset_id": source["dataset_id"],
            "source_dataset_version": remote[name]["version"],
            "retrieved_utc": retrieved_utc.isoformat(),
            "archive_note": "First retrieved copy; never overwritten",
        })
        for v in field.variables:
            field[v].encoding = {}

        # Write to a temporary name first, so an interrupted run never
        # leaves a half-written file that looks complete.
        partial = path.with_name(path.name + ".part")
        field.to_netcdf(partial, encoding={variable: {"zlib": True, "complevel": 4}})
        partial.rename(path)

        rows.append({**row, "Status": "Saved", "Valid pixels": valid})

log = pd.DataFrame(rows)
log["Retrieved (UTC)"] = retrieved_utc.isoformat()

log_path = ARCHIVE_DIR / "archive_log.csv"
log.to_csv(log_path, mode="a", header=not log_path.exists(), index=False)

display(log.groupby(["Source", "Status"]).size().unstack(fill_value=0))
display(log[log["Status"] == "Saved"])


Status,Saved
Source,
chl,9
chl_l4,12
sst,30


,Source,Date,Status,Valid pixels,Retrieved (UTC)
0,chl,2026-10-10,Saved,699,2026-10-19T15:17:38.709187+00:00
1,chl,2026-10-11,Saved,11674,2026-10-19T15:17:38.709187+00:00
2,chl,2026-10-12,Saved,5984,2026-10-19T15:17:38.709187+00:00
3,chl,2026-10-13,Saved,11661,2026-10-19T15:17:38.709187+00:00
4,chl,2026-10-14,Saved,12132,2026-10-19T15:17:38.709187+00:00
5,chl,2026-10-15,Saved,12095,2026-10-19T15:17:38.709187+00:00
6,chl,2026-10-16,Saved,12190,2026-10-19T15:17:38.709187+00:00
7,chl,2026-10-17,Saved,6941,2026-10-19T15:17:38.709187+00:00
8,chl,2026-10-18,Saved,10901,2026-10-19T15:17:38.709187+00:00
9,chl_l4,2026-10-07,Saved,12380,2026-10-19T15:17:38.709187+00:00


In [4]:
rows = []

for name in SOURCES:
    files = sorted((ARCHIVE_DIR / name).glob("*.nc"))
    if not files:
        continue

    dates = pd.DatetimeIndex([pd.Timestamp(p.stem) for p in files])
    missing = pd.date_range(dates.min(), dates.max(), freq="D").difference(dates)

    rows.append({
        "Source": name,
        "Days archived": len(dates),
        "First": f"{dates.min():%Y-%m-%d}",
        "Last": f"{dates.max():%Y-%m-%d}",
        "Missing days": ", ".join(f"{d:%Y-%m-%d}" for d in missing) or "None",
    })

display(pd.DataFrame(rows).set_index("Source"))


,Days archived,First,Last,Missing days
Source,,,,
chl,9,2026-10-10,2026-10-18,None
chl_l4,12,2026-10-07,2026-10-18,None
sst,30,2026-09-20,2026-10-19,None
